# Data Preprocessing
---
**Objectives:**
- Remove erroneous rows identified in EDA
- Impute missing values with appropriate strategies
- Encode categorical variables correctly (ordinal vs nominal)
- Split data into train/test with stratification
- Scale numerical features (fit on train only — no leakage)
- Save processed splits for downstream notebooks

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ── Constants ──
RAW_DATA_PATH  = '../data/raw/credit_risk_dataset.csv'
PROCESSED_DIR  = Path('../data/processed')
RANDOM_STATE   = 42
TEST_SIZE      = 0.2
TARGET_COL     = 'loan_status'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 4)})

print('Setup complete.')

Setup complete.


## 1. Load Data

In [2]:
raw_df = pd.read_csv(RAW_DATA_PATH)
df     = raw_df.copy()   # always preserve the original

print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Loaded: 32,581 rows × 12 columns


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3


## 2. Remove Erroneous Rows

From EDA we identified two columns with biologically/logically impossible values that are data entry errors, not real outliers to cap:
- `person_age > 100`: cannot be a valid loan applicant age
- `person_emp_length > 60`: cannot exceed a realistic working life

In [3]:
before = len(df)

df = df[df['person_age'] <= 100]
df = df[df['person_emp_length'] <= 60]

removed = before - len(df)
print(f'Rows removed: {removed} ({removed / before * 100:.2f}%)')
print(f'Remaining rows: {len(df):,}')

Rows removed: 902 (2.77%)
Remaining rows: 31,679


## 3. Handle Missing Values

In [4]:
# Confirm missing state before imputation
print('Missing values before imputation:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values before imputation:
loan_int_rate    3047
dtype: int64


### 3a. `person_emp_length` - Median Imputation
Employment length is right-skewed, so median is more robust than mean.

In [6]:
emp_median = df['person_emp_length'].median()
df['person_emp_length'] = df['person_emp_length'].fillna(emp_median)
print(f'person_emp_length imputed with median: {emp_median}')

person_emp_length imputed with median: 4.0


### 3b. `loan_int_rate` - Group-wise Median by `loan_grade`
Interest rate is strongly determined by loan grade (Grade A -> lowest rate, Grade G → highest).  
Imputing with the overall median would ignore this structure, group-wise median is more accurate.

In [7]:
grade_median_rates = df.groupby('loan_grade')['loan_int_rate'].median()
print('Median interest rate by loan grade:')
print(grade_median_rates)

df['loan_int_rate'] = df.apply(
    lambda row: grade_median_rates[row['loan_grade']]
    if pd.isnull(row['loan_int_rate']) else row['loan_int_rate'],
    axis=1
)

Median interest rate by loan grade:
loan_grade
A     7.490
B    10.990
C    13.480
D    15.310
E    16.795
F    18.530
G    20.160
Name: loan_int_rate, dtype: float64


In [8]:
# Verify no missing values remain
assert df.isnull().sum().sum() == 0, 'Missing values still present!'
print('No missing values remaining.')

No missing values remaining.


## 4. Encode Categorical Variables

### 4a. `loan_grade` - Ordinal Encoding
`loan_grade` has a natural order (A is best, G is worst creditworthiness). One-hot encoding would destroy this ordinal relationship. We map to integers that preserve the rank.

In [9]:
GRADE_ORDER = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['loan_grade'] = df['loan_grade'].map(GRADE_ORDER)
print('loan_grade encoded:', df['loan_grade'].value_counts().sort_index().to_dict())

loan_grade encoded: {1: 10370, 2: 10183, 3: 6319, 4: 3555, 5: 952, 6: 236, 7: 64}


### 4b. `cb_person_default_on_file` - Binary Encoding
This is a simple Yes/No flag, map directly to 1/0.

In [10]:
df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
print('cb_person_default_on_file encoded:', df['cb_person_default_on_file'].value_counts().to_dict())

cb_person_default_on_file encoded: {0: 26051, 1: 5628}


### 4c. `person_home_ownership` & `loan_intent` - One-Hot Encoding
Nominal categories with no intrinsic order. We drop the first dummy to avoid multicollinearity (dummy variable trap).

In [11]:
NOMINAL_COLS = ['person_home_ownership', 'loan_intent']

df = pd.get_dummies(df, columns=NOMINAL_COLS, drop_first=True, dtype=int)

print(f'Shape after encoding: {df.shape}')
print('New columns added:')
print([c for c in df.columns if any(n in c for n in NOMINAL_COLS)])

Shape after encoding: (31679, 18)
New columns added:
['person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']


## 5. Train-Test Split

> **Important:** We split *before* scaling. The scaler must be fit only on training data - fitting it on the full dataset would leak test set statistics into training, giving artificially optimistic results.

In [12]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,              # preserves 78/22 class ratio in both splits
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows')
print(f'\nTrain class distribution:')
print(y_train.value_counts(normalize=True).mul(100).round(2))
print(f'\nTest class distribution:')
print(y_test.value_counts(normalize=True).mul(100).round(2))

Train: 25,343 rows | Test: 6,336 rows

Train class distribution:
loan_status
0    78.46
1    21.54
Name: proportion, dtype: float64

Test class distribution:
loan_status
0    78.46
1    21.54
Name: proportion, dtype: float64


Both splits maintain the ~78/22 class ratio, stratification worked correctly.

## 6. Feature Scaling

We apply `StandardScaler` to numerical columns only. Encoded categoricals (0/1 dummies, ordinal grade) do not need scaling.  

> **Leakage prevention:** `.fit()` only on `X_train`. We then `.transform()` both train and test using the same parameters learned from train.

In [14]:
NUMERICAL_COLS = [
    'person_age', 'person_income', 'person_emp_length',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_cred_hist_length'
]

scaler = StandardScaler()

X_train[NUMERICAL_COLS] = scaler.fit_transform(X_train[NUMERICAL_COLS])   # fit + transform on train
X_test[NUMERICAL_COLS]  = scaler.transform(X_test[NUMERICAL_COLS])         # transform only on test

print('Scaling complete.')
print('\nTrain numerical stats after scaling (should be ~0 mean, ~1 std):')
X_train[NUMERICAL_COLS].describe().loc[['mean', 'std']].round(4)

Scaling complete.

Train numerical stats after scaling (should be ~0 mean, ~1 std):


,person_age,person_income,person_emp_length,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length
mean,-0.0,-0.0,-0.0,-0.0,0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## 7. Final Sanity Check

In [16]:
print('=== Final Dataset Summary ===')
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}  | Default rate: {y_train.mean():.3f}')
print(f'y_test  : {y_test.shape}   | Default rate: {y_test.mean():.3f}')
print(f'\nFeatures ({X_train.shape[1]}):')
print(list(X_train.columns))

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0
print('\nNo nulls in processed data. All checks passed.')

=== Final Dataset Summary ===
X_train : (25343, 17)
X_test  : (6336, 17)
y_train : (25343,)  | Default rate: 0.215
y_test  : (6336,)   | Default rate: 0.215

Features (17):
['person_age', 'person_income', 'person_emp_length', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'person_home_ownership_OTHER', 'person_home_ownership_OWN', 'person_home_ownership_RENT', 'loan_intent_EDUCATION', 'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL', 'loan_intent_PERSONAL', 'loan_intent_VENTURE']

No nulls in processed data. All checks passed.


## 8. Save Processed Splits

In [17]:
X_train.to_csv(PROCESSED_DIR / 'X_train.csv', index=False)
X_test.to_csv(PROCESSED_DIR  / 'X_test.csv',  index=False)
y_train.to_csv(PROCESSED_DIR / 'y_train.csv', index=False)
y_test.to_csv(PROCESSED_DIR  / 'y_test.csv',  index=False)

print('Saved to data/processed/:')
for f in PROCESSED_DIR.iterdir():
    print(f'  {f.name}')

Saved to data/processed/:
  X_test.csv
  y_test.csv
  test.csv
  train.csv
  X_train.csv
  y_train.csv


## 9. Preprocessing Summary

| Step | What was done | Rationale |
|---|---|---|
| **Error removal** | Dropped rows with `age > 100` or `emp_length > 60` | Data entry errors, not real outliers |
| **Missing - emp_length** | Median imputation | Right-skewed; median is robust |
| **Missing - int_rate** | Group-wise median by `loan_grade` | Rate is grade-dependent; global median would be inaccurate |
| **Ordinal encoding** | `loan_grade` mapped A=1 … G=7 | Ordinal relationship is meaningful |
| **Binary encoding** | `cb_person_default_on_file` -> 0/1 | Direct Y/N map |
| **One-hot encoding** | `home_ownership`, `loan_intent` | Nominal -> no inherent order |
| **Train-test split** | 80/20, stratified on `loan_status` | Preserves class ratio; stratify avoids imbalance skew |
| **Scaling** | StandardScaler on numerical cols, fit on train only | Prevents data leakage from test set |
